In [148]:
import pandas as pd
import numpy as np

In [149]:
player_full_df = pd.read_csv(
    "../data/processed/player_games.csv"
)

In [150]:
rating_df = player_full_df.copy()

In [151]:
rating_df["twoPointersMade"] = (
    rating_df["fieldGoalsMade"]
    - rating_df["threePointersMade"]
)

rating_df["twoPointersAttempted"] = (
    rating_df["fieldGoalsAttempted"]
    - rating_df["threePointersAttempted"]
)

rating_df["twoPointPercentage"] = (
    rating_df["twoPointersMade"]
    / rating_df["twoPointersAttempted"].replace(0, np.nan)
)

In [152]:
rating_df["minutes"].dtype

<StringDtype(storage='python', na_value=nan)>

In [153]:
rating_df["minutes"].head(10)

0    45:15
1    38:30
2    36:51
3    41:39
4    47:13
5    12:53
6    20:32
7    30:11
8     2:15
9    14:36
Name: minutes, dtype: str

In [154]:
def convert_minutes(value):
    if pd.isna(value):
        return np.nan

    minutes, seconds = value.split(":")
    return int(minutes) + int(seconds) / 60

In [155]:
rating_df["minutesNumeric"] = (
    rating_df["minutes"].apply(convert_minutes)
)

In [156]:
rating_df[
    ["minutes", "minutesNumeric"]
].head(10)

,minutes,minutesNumeric
0,45:15,45.250000
1,38:30,38.500000
2,36:51,36.850000
3,41:39,41.650000
4,47:13,47.216667
5,12:53,12.883333
6,20:32,20.533333
7,30:11,30.183333
8,2:15,2.250000
9,14:36,14.600000


In [157]:
rating_df = rating_df[
    rating_df["minutesNumeric"] > 0
].copy()

In [158]:
weighted_cols = [
    "usagePercentage",
    "assistPercentage",
    "assistToTurnover",
    "assistRatio",
    "turnoverRatio",
    "offensiveReboundPercentage",
    "defensiveReboundPercentage",
    "reboundPercentage",
    "offensiveRating",
    "defensiveRating",
    "netRating",
    "PIE",
]

for col in weighted_cols:
    rating_df[f"{col}Weighted"] = (
        rating_df[col] * rating_df["minutesNumeric"]
    )

In [159]:
player_profile_df = (
    rating_df
    .groupby(
        ["personId", "firstName", "familyName"],
        as_index=False
    )
    .agg(
        gamesPlayed=("gameId", "nunique"),
        totalMinutes=("minutesNumeric", "sum"),

        # scoring volume
        points=("points", "sum"),

        # shooting totals
        fieldGoalsMade=("fieldGoalsMade", "sum"),
        fieldGoalsAttempted=("fieldGoalsAttempted", "sum"),
        threePointersMade=("threePointersMade", "sum"),
        threePointersAttempted=("threePointersAttempted", "sum"),
        freeThrowsMade=("freeThrowsMade", "sum"),
        freeThrowsAttempted=("freeThrowsAttempted", "sum"),
        twoPointersMade=("twoPointersMade", "sum"),
        twoPointersAttempted=("twoPointersAttempted", "sum"),

        assists=("assists", "sum"),
        turnovers=("turnovers", "sum"),
        steals=("steals", "sum"),
        blocks=("blocks", "sum"),
        reboundsOffensive=("reboundsOffensive", "sum"),
        reboundsDefensive=("reboundsDefensive", "sum"),
        reboundsTotal=("reboundsTotal", "sum"),

        usagePercentageWeighted=("usagePercentageWeighted", "sum"),
        assistPercentageWeighted=("assistPercentageWeighted", "sum"),
        assistToTurnoverWeighted=("assistToTurnoverWeighted", "sum"),
        assistRatioWeighted=("assistRatioWeighted", "sum"),
        turnoverRatioWeighted=("turnoverRatioWeighted", "sum"),

        offensiveReboundPercentageWeighted=("offensiveReboundPercentageWeighted", "sum"),
        defensiveReboundPercentageWeighted=("defensiveReboundPercentageWeighted", "sum"),
        reboundPercentageWeighted=("reboundPercentageWeighted", "sum"),

        offensiveRatingWeighted=("offensiveRatingWeighted", "sum"),
        defensiveRatingWeighted=("defensiveRatingWeighted", "sum"),
        netRatingWeighted=("netRatingWeighted", "sum"),
        PIEWeighted=("PIEWeighted", "sum"),
    )
)

In [160]:
player_profile_df["twoPointPercentage"] = (
    player_profile_df["twoPointersMade"]
    / player_profile_df["twoPointersAttempted"].replace(0, np.nan)
)

player_profile_df["threePointersPercentage"] = (
    player_profile_df["threePointersMade"]
    / player_profile_df["threePointersAttempted"].replace(0, np.nan)
)

player_profile_df["fieldGoalPercentage"] = (
    player_profile_df["fieldGoalsMade"]
    / player_profile_df["fieldGoalsAttempted"].replace(0, np.nan)
)

player_profile_df["freeThrowPercentage"] = (
    player_profile_df["freeThrowsMade"]
    / player_profile_df["freeThrowsAttempted"].replace(0, np.nan)
)

player_profile_df["pointsPerMinute"] = (
    player_profile_df["points"]
    / player_profile_df["totalMinutes"]
)

player_profile_df["assistsPerMinute"] = (
    player_profile_df["assists"]
    / player_profile_df["totalMinutes"]
)

player_profile_df["stealsPerMinute"] = (
    player_profile_df["steals"]
    / player_profile_df["totalMinutes"]
)

player_profile_df["blocksPerMinute"] = (
    player_profile_df["blocks"]
    / player_profile_df["totalMinutes"]
)

player_profile_df["threePointAttemptRate"] = (
    player_profile_df["threePointersAttempted"]
    / player_profile_df["fieldGoalsAttempted"].replace(0, np.nan)
)

player_profile_df["freeThrowRate"] = (
    player_profile_df["freeThrowsAttempted"]
    / player_profile_df["fieldGoalsAttempted"].replace(0, np.nan)
)

player_profile_df["trueShootingPercentage"] = (
    player_profile_df["points"]
    / (
        2 * (
            player_profile_df["fieldGoalsAttempted"]
            + 0.44 * player_profile_df["freeThrowsAttempted"]
        )
    ).replace(0, np.nan)
)

for col in weighted_cols:
    player_profile_df[col] = (
        player_profile_df[f"{col}Weighted"]
        / player_profile_df["totalMinutes"]
    )

In [161]:
player_profile_df.head()

,personId,firstName,familyName,gamesPlayed,totalMinutes,points,fieldGoalsMade,fieldGoalsAttempted,threePointersMade,threePointersAttempted,...,assistToTurnover,assistRatio,turnoverRatio,offensiveReboundPercentage,defensiveReboundPercentage,reboundPercentage,offensiveRating,defensiveRating,netRating,PIE
0,201142,Kevin,Durant,1,47.050000,23,9,16,0,4,...,0.75,12.0,16.0,0.000,0.167,0.090,107.6,110.0,-2.4,0.121
1,201143,Al,Horford,1,20.250000,5,2,7,1,4,...,0.50,10.0,20.0,0.000,0.385,0.161,104.7,122.2,-17.6,0.000
2,201144,Mike,Conley,1,12.716667,3,1,5,1,2,...,0.00,0.0,16.7,0.067,0.100,0.080,100.0,123.1,-23.1,-0.037
3,201566,Russell,Westbrook,1,18.800000,6,2,8,0,2,...,0.50,8.3,16.7,0.056,0.227,0.150,121.1,110.0,11.1,0.052
4,201939,Stephen,Curry,1,32.066667,23,6,14,3,9,...,2.00,17.4,8.7,0.000,0.038,0.018,112.9,108.3,4.5,0.135


In [162]:
player_profile_df[
    [
        "firstName",
        "familyName",
        "gamesPlayed",
        "totalMinutes",
        "twoPointPercentage",
        "threePointersPercentage",
        "pointsPerMinute",
        "assistsPerMinute",
        "reboundPercentage",
        "defensiveRating",
    ]
].head(20)

,firstName,familyName,gamesPlayed,totalMinutes,twoPointPercentage,threePointersPercentage,pointsPerMinute,assistsPerMinute,reboundPercentage,defensiveRating
0,Kevin,Durant,1,47.050000,0.750000,0.000000,0.488842,0.063762,0.090,110.0
1,Al,Horford,1,20.250000,0.333333,0.250000,0.246914,0.049383,0.161,122.2
2,Mike,Conley,1,12.716667,0.000000,0.500000,0.235911,0.000000,0.080,123.1
3,Russell,Westbrook,1,18.800000,0.333333,0.000000,0.319149,0.053191,0.150,110.0
4,Stephen,Curry,1,32.066667,0.600000,0.333333,0.717256,0.124740,0.018,108.3
5,DeMar,DeRozan,1,37.416667,0.733333,0.500000,0.775056,0.240535,0.075,107.2
6,Jrue,Holiday,1,33.066667,0.500000,0.142857,0.423387,0.211694,0.085,102.7
7,Bismack,Biyombo,1,4.933333,NaN,NaN,0.000000,0.000000,0.143,90.9
8,Klay,Thompson,1,22.333333,0.375000,0.200000,0.447761,0.044776,0.043,133.3
9,Jimmy,Butler III,1,34.733333,0.500000,0.500000,0.892514,0.115163,0.082,104.1


In [163]:
def percentile_score(series):
    return series.rank(pct=True) * 100

def weighted_rating(components):
    numerator = 0
    denominator = 0

    for series, weight in components:
        valid = series.notna()

        numerator += series.fillna(0) * weight
        denominator += valid.astype(float) * weight

    return numerator / denominator.replace(0, np.nan)

In [164]:
player_profile_df["finishing"] = weighted_rating([
    (
        percentile_score(
            player_profile_df["twoPointPercentage"]
        ),
        0.40
    ),
    (
        percentile_score(
            player_profile_df["freeThrowRate"]
        ),
        0.25
    ),
    (
        percentile_score(
            player_profile_df["pointsPerMinute"]
        ),
        0.20
    ),
    (
        percentile_score(
            player_profile_df["trueShootingPercentage"]
        ),
        0.15
    ),
])

player_profile_df["shooting"] = weighted_rating([
    (
        percentile_score(
            player_profile_df["threePointersPercentage"]
        ),
        0.40
    ),
    (
        percentile_score(
            player_profile_df["threePointAttemptRate"]
        ),
        0.25
    ),
    (
        percentile_score(
            player_profile_df["trueShootingPercentage"]
        ),
        0.20
    ),
    (
        percentile_score(
            player_profile_df["freeThrowPercentage"]
        ),
        0.15
    ),
])

player_profile_df["playmaking"] = weighted_rating([
    (
        percentile_score(
            player_profile_df["assistPercentage"]
        ),
        0.40
    ),
    (
        percentile_score(
            player_profile_df["assistToTurnover"]
        ),
        0.25
    ),
    (
        percentile_score(
            player_profile_df["assistRatio"]
        ),
        0.20
    ),
    (
        100 - percentile_score(
            player_profile_df["turnoverRatio"]
        ),
        0.15
    ),
])

player_profile_df["rebounding"] = weighted_rating([
    (
        percentile_score(
            player_profile_df["reboundPercentage"]
        ),
        0.45
    ),
    (
        percentile_score(
            player_profile_df["offensiveReboundPercentage"]
        ),
        0.30
    ),
    (
        percentile_score(
            player_profile_df["defensiveReboundPercentage"]
        ),
        0.25
    ),
])

player_profile_df["defense"] = weighted_rating([
    (
        percentile_score(
            player_profile_df["stealsPerMinute"]
        ),
        0.30
    ),
    (
        percentile_score(
            player_profile_df["blocksPerMinute"]
        ),
        0.30
    ),
    (
        100 - percentile_score(
            player_profile_df["defensiveRating"]
        ),
        0.40
    ),
])

In [165]:
ratings_df = player_profile_df[
    [
        "personId",
        "firstName",
        "familyName",
        "gamesPlayed",
        "totalMinutes",
        "finishing",
        "shooting",
        "playmaking",
        "defense",
        "rebounding",
    ]
].copy()

ratings_df[
    [
        "finishing",
        "shooting",
        "playmaking",
        "defense",
        "rebounding",
    ]
].isna().sum()

finishing      0
shooting      14
playmaking     0
defense        0
rebounding     0
dtype: int64

In [166]:
rating_cols = [
    "finishing",
    "shooting",
    "playmaking",
    "defense",
    "rebounding",
]

ratings_df[rating_cols] = (
    ratings_df[rating_cols]
    .round(1)
)

In [167]:
ratings_df.sort_values(
    "shooting",
    ascending=False
).head(20)

,personId,firstName,familyName,gamesPlayed,totalMinutes,finishing,shooting,playmaking,defense,rebounding
147,1631127,Harrison,Ingram,1,4.066667,61.5,98.2,25.5,55.4,72.4
144,1631108,Max,Christie,1,25.950000,46.4,96.1,42.4,44.5,11.2
201,1642856,Egor,Dëmin,1,22.316667,74.6,91.8,57.9,46.8,60.5
200,1642851,Kon,Knueppel,1,25.483333,47.4,91.5,34.6,50.3,51.8
210,1642954,Will,Richard,1,13.583333,65.4,89.1,59.3,36.6,42.9
68,1629013,Landry,Shamet,1,13.816667,53.2,88.3,55.9,63.9,39.2
91,1629731,Dean,Wade,1,28.116667,74.0,85.9,25.5,67.7,22.9
80,1629611,Terance,Mann,1,19.450000,74.5,83.3,45.5,37.3,30.0
112,1630540,Miles,McBride,1,25.866667,49.5,83.2,36.1,63.0,32.5
76,1629060,Rui,Hachimura,1,35.416667,38.1,82.9,65.9,28.7,38.2
